## librarys imports

In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import re
import sys
from pathlib import Path


### data stracting

In [4]:

sys.path.append(str(Path().resolve().parent))

import scripts.fetch_steamspy as sp

from scripts.fetch_steamspy import fetch_steamspy_data

df = fetch_steamspy_data()
df.head()

Data saved at: D:\Proyectos\SteamCB\data\raw\steamspy_games.csv


,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
730,730,Counter-Strike: Global Offensive,Valve,Valve,,7642084,1173003,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,1013936
1172470,1172470,Apex Legends,Respawn,Electronic Arts,,668053,326926,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,124262
578080,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",,1520457,1037487,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,314682
1623730,1623730,Palworld,Pocketpair,Pocketpair,,358266,22443,0,"50,000,000 .. 100,000,000",0,0,0,0,2999,2999,0,18028
440,440,Team Fortress 2,Valve,Valve,,1044264,117208,0,"50,000,000 .. 100,000,000",0,0,0,0,0,0,0,43819


In [7]:
df.columns

Index(['appid', 'name', 'developer', 'publisher', 'score_rank', 'positive',
       'negative', 'userscore', 'owners', 'average_forever', 'average_2weeks',
       'median_forever', 'median_2weeks', 'price', 'initialprice', 'discount',
       'ccu'],
      dtype='str')

## Tables Infrasture


## Games table


In [8]:


df_games = pd.concat(
    [
        df[['appid', 'name','developer', 'owners']],  
        df.loc[:, "price":"ccu"]
    ],
    axis=1
)

df_games.head()

,appid,name,developer,owners,price,initialprice,discount,ccu
730,730,Counter-Strike: Global Offensive,Valve,"100,000,000 .. 200,000,000",0,0,0,1013936
1172470,1172470,Apex Legends,Respawn,"100,000,000 .. 200,000,000",0,0,0,124262
578080,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"100,000,000 .. 200,000,000",0,0,0,314682
1623730,1623730,Palworld,Pocketpair,"50,000,000 .. 100,000,000",2999,2999,0,18028
440,440,Team Fortress 2,Valve,"50,000,000 .. 100,000,000",0,0,0,43819


In [9]:
df_games.info()

<class 'pandas.DataFrame'>
Index: 1000 entries, 730 to 1150690
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   appid         1000 non-null   int64
 1   name          1000 non-null   str  
 2   developer     1000 non-null   str  
 3   owners        1000 non-null   str  
 4   price         1000 non-null   str  
 5   initialprice  1000 non-null   str  
 6   discount      1000 non-null   str  
 7   ccu           1000 non-null   int64
dtypes: int64(2), str(6)
memory usage: 70.3+ KB


### Calculate average owners, because Steam provide a range.


In [10]:

def owners_to_mean(x):
    numbers = re.findall(r'\d+', x) # loock for each number
    if len(numbers)  != 2:
        return None
    min_val, max_val = map(int,numbers)
    return(min_val + max_val)/2

df_games['owners_mean'] = df_games['owners'].apply(owners_to_mean)

### Convert str data to integer type.

In [11]:
df_games[['discount','price']] = df_games[['discount','price']].astype(int)


### Changes

- **`initial_price`**: Removed to avoid possible **multicollinearity** issues with other related columns (for example, `price` and `discount`).  
- **`owners`**: Removed because the **`owners_mean`** column is available, which summarizes the information in a more useful way for the analysis.

In [12]:
df_games = df_games.drop(['owners', 'initialprice'], axis=1)
df_games.info()

<class 'pandas.DataFrame'>
Index: 1000 entries, 730 to 1150690
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   appid        1000 non-null   int64 
 1   name         1000 non-null   str   
 2   developer    1000 non-null   str   
 3   price        1000 non-null   int64 
 4   discount     1000 non-null   int64 
 5   ccu          1000 non-null   int64 
 6   owners_mean  0 non-null      object
dtypes: int64(4), object(1), str(2)
memory usage: 62.5+ KB


In [13]:
## Save
df_games.to_csv("table_games.csv", index=False)

## Review table

In [14]:
df_review = df[[
    'appid', 
    'positive', 
    'negative', 
    'median_forever'
    ]]

df_review.info()

<class 'pandas.DataFrame'>
Index: 1000 entries, 730 to 1150690
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   appid           1000 non-null   int64
 1   positive        1000 non-null   int64
 2   negative        1000 non-null   int64
 3   median_forever  1000 non-null   int64
dtypes: int64(4)
memory usage: 39.1+ KB


## column review_sentiment 
This column allows us to determine whether the reviews are mostly positive or negative.

In [15]:
df_review['review_sentiment'] = df_review.apply(
    lambda row: 'positive' if row['positive'] >= row['negative'] else 'negative',
    axis=1
)

### Score column
Provides a score based on user evaluations

In [16]:
df_review['score'] = pd.cut(
    df_review['positive'] / (df_review['positive'] + df_review['negative']),
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

### Changes
- Drop negative and positive columns because review_sentiment provides a better representation of reviews.

In [17]:
df_review = df_review.drop(['positive','negative'], axis= 1)
df_review.info()

<class 'pandas.DataFrame'>
Index: 1000 entries, 730 to 1150690
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   appid             1000 non-null   int64
 1   median_forever    1000 non-null   int64
 2   review_sentiment  1000 non-null   str  
 3   score             1000 non-null   int64
dtypes: int64(3), str(1)
memory usage: 39.1+ KB


In [18]:
#Save
df_review.to_csv("table_review.csv", index=False)